# SteerViT: Steerable Visual Representations

This notebook provides two interactive demos to showcase the capabilities of SteerViT. For more information and further use-cases, visit the [Project Website](https://jonaruthardt.github.io/project/SteerViT/).

In [11]:
print("Loading SteerViT demo...")

try:
    from steervit import SteerViT
except ImportError as e:
    print(f"Error importing SteerViT: {e}")
    print("Please install SteerViT using the following command:")
    print(" pip install git+https://github.com/JonaRuthardt/SteerViT.git")

import os, sys
import threading

from PIL import Image
import requests
from io import BytesIO
from collections import defaultdict

from datasets import load_dataset

from timm.data import resolve_data_config, create_transform
import torch
import torch.nn.functional as F
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import matplotlib.pyplot as plt

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

Loading SteerViT demo...


In [12]:
CHECKPOINT_OPTIONS = {
    "SteerDINOv2-Base": "steervit_dinov2_base.pth",
    "SteerMAE-Base": "steervit_mae_base.pth",
}

DEFAULT_CHECKPOINT_NAME = "SteerDINOv2-Base"
SELECTED_CHECKPOINT_NAME = DEFAULT_CHECKPOINT_NAME
SELECTED_CHECKPOINT_PATH = CHECKPOINT_OPTIONS[SELECTED_CHECKPOINT_NAME]

model = None
transforms = None
attention_demo_controls = None
retrieval_controls = None

raw_transforms = create_transform(
        input_size=(336,336),
        is_training=False,
        color_jitter=0.0,
        auto_augment=None,
        interpolation="bicubic",
        crop_mode="center",
        normalize=False,
    )

def set_selected_checkpoint(checkpoint_name: str):
    global SELECTED_CHECKPOINT_NAME, SELECTED_CHECKPOINT_PATH
    if checkpoint_name not in CHECKPOINT_OPTIONS:
        raise ValueError(f"Unknown checkpoint: {checkpoint_name}")
    SELECTED_CHECKPOINT_NAME = checkpoint_name
    SELECTED_CHECKPOINT_PATH = CHECKPOINT_OPTIONS[checkpoint_name]

def load_selected_model():
    global model, transforms
    model = SteerViT.from_pretrained(SELECTED_CHECKPOINT_PATH).to(device)
    transforms = model.get_transforms()
    return model, transforms

if WIDGETS_AVAILABLE:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    checkpoint_dropdown = widgets.Dropdown(
        options=list(CHECKPOINT_OPTIONS.keys()),
        value=DEFAULT_CHECKPOINT_NAME,
        description="Checkpoint:",
        layout=widgets.Layout(width="500px"),
        style={"description_width": "90px"},
    )

    load_button = widgets.Button(
        description="Load Model",
        button_style="primary",
        tooltip="Load the selected checkpoint",
        icon="download",
    )

    output = widgets.Output()

    def on_checkpoint_change(change):
        if change["name"] == "value":
            set_selected_checkpoint(change["new"])

    def on_load_button_click(_):
        with output:
            clear_output()
            print(f"Loading model from: {SELECTED_CHECKPOINT_PATH}")
            try:
                load_button.disabled = True
                load_button.description = "Loading..."
                load_selected_model()
                print("Model loaded successfully.")
                print(f"Selected checkpoint: {SELECTED_CHECKPOINT_NAME}")
                print(f"Checkpoint path: {SELECTED_CHECKPOINT_PATH}")
            except Exception as e:
                print(f"Failed to load model: {e}")
            finally:
                load_button.disabled = False
                load_button.description = "Load Model"

    checkpoint_dropdown.observe(on_checkpoint_change, names="value")
    load_button.on_click(on_load_button_click)

    display(widgets.VBox([
        widgets.HTML("<h2>Checkpoint Selection</h2>"),
        checkpoint_dropdown,
        load_button,
        output,
    ]))
else:
    print("ipywidgets is not available.")
    print("Using default checkpoint:")
    print(f"  {SELECTED_CHECKPOINT_NAME}")
    print(f"  {SELECTED_CHECKPOINT_PATH}")

    # Optional manual load in non-widget mode:
    model, transforms = load_selected_model()

In [13]:


DEFAULT_IMAGE_URL = (
    "https://hips.hearstapps.com/clv.h-cdn.co/assets/16/18/gettyimages-586890581.jpg?crop=0.668xw:1.00xh;0.219xw,0"
)
DEFAULT_PROMPT = "the dog"



def download_image(image_url: str) -> Image.Image:
    try:
        response = requests.get(image_url)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert("RGB")
        image = raw_transforms(image)
        image = Image.fromarray((image.permute(1, 2, 0).numpy()).astype("uint8"))
        return image
    except Exception as e:
        print(f"Error downloading image: {e}")
        raise

def run_attention_demo(image_url: str, prompt: str):
    assert model is not None, "Model is not loaded. Please load the model before proceeding."
    
    # Load and preprocess the image
    image = download_image(image_url)
    input_tensor = transforms(image).unsqueeze(0)

    heatmap = model.get_attention_heatmaps(input_tensor, [prompt]).squeeze(0).cpu().numpy()
    # base_heatmap = model.get_base_attention_heatmaps(input_tensor).squeeze(0).cpu().numpy()
    base_heatmap = model.get_attention_heatmaps(input_tensor).squeeze(0).cpu().numpy()

    fig, ax = plt.subplots(figsize=(4 * 3, 4), ncols=3)
    ax[0].set_title("Input Image")
    ax[0].imshow(image)
    if "dinov2" in SELECTED_CHECKPOINT_NAME.lower():
        ax[1].set_title("DINOv2 (Prompt-Agnostic)")
    elif "mae" in SELECTED_CHECKPOINT_NAME.lower():
        ax[1].set_title("MAE (Prompt-Agnostic)")
    else:
        ax[1].set_title("Base ViT (Prompt-Agnostic)")
    # ax[1].set_title(f"{model.config['vision_encoder']['model_name']} (Prompt-Agnostic)")
    ax[1].imshow(image)
    ax[1].imshow(base_heatmap, cmap="jet", alpha=0.5)
    ax[2].set_title("SteerViT (Prompt-Aware)")
    ax[2].imshow(image)
    ax[2].imshow(heatmap, cmap="jet", alpha=0.5)

    for a in ax:
        a.axis("off")

    plt.show()
    plt.close(fig)

def build_demo_ui():
    global attention_demo_controls

    if not WIDGETS_AVAILABLE:
        print("ipywidgets is not available in this environment.")
        print("Use the function directly:")
        print('run_attention_demo(image_url="", prompt="")')
        return

    if attention_demo_controls is not None:
        attention_demo_controls.close()

    title = widgets.HTML(
        value="<h2>CLS Attention Heatmap Visualization</h2>"
    )

    image_url_input = widgets.Textarea(
        value=DEFAULT_IMAGE_URL,
        description="Image URL:",
        layout=widgets.Layout(width="900px", height="40px"),
        style={"description_width": "90px"},
        placeholder="Paste an image URL here..."
    )

    prompt_input = widgets.Text(
        value=DEFAULT_PROMPT,
        description="Prompt:",
        layout=widgets.Layout(width="900px"),
        style={"description_width": "90px"},
        placeholder="Enter a text prompt..."
    )

    run_button = widgets.Button(
        description="Generate heatmap",
        button_style="primary",
        icon="fire"
    )

    reset_button = widgets.Button(
        description="Reset defaults",
        button_style="",
        icon="refresh"
    )

    output = widgets.Output()

    def on_run_clicked(_):
        with output:
            clear_output(wait=True)
            try:
                run_attention_demo(
                    image_url=image_url_input.value,
                    prompt=prompt_input.value,
                )
            except Exception as e:
                print(f"Error: {e}")

    def on_reset_clicked(_):
        image_url_input.value = DEFAULT_IMAGE_URL
        prompt_input.value = DEFAULT_PROMPT

    run_button.on_click(on_run_clicked)
    reset_button.on_click(on_reset_clicked)

    attention_demo_controls = widgets.VBox([
        title,
        image_url_input,
        prompt_input,
        widgets.HBox([run_button, reset_button]),
        output,
    ])

    display(attention_demo_controls)


build_demo_ui()

In [14]:
DEFAULT_COCO_INDEX = 4486
DEFAULT_COCO_PROMPT = "the remote control"
COCO_ROOT = "coco_val_only"
COCO_SPLIT = "val2017"
COCO_BATCH_SIZE = 16

### Load COCO dataset for retrieval demo ### 

from torchvision.datasets import CocoDetection
from torchvision.datasets.utils import download_and_extract_archive

root = "./coco_val_only"

download_and_extract_archive(
    "http://images.cocodataset.org/zips/val2017.zip",
    download_root=root,
    extract_root=root,
)

class SimpleCOCO(torch.utils.data.Dataset):
    def __init__(self, root, split="val2017", transform=None):
        self.image_files = [f"{root}/{split}/{img}" for img in sorted(os.listdir(f"{root}/{split}"))]
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image = Image.open(self.image_files[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image
    
    def get_image(self, idx):
        return Image.open(self.image_files[idx]).convert("RGB")
    
dataset = SimpleCOCO(root="coco_val_only", split="val2017", transform=transforms)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=False)

stop_state = {"stop": False}
run_state = {"running": False, "thread": None}


def extract_base_features(batch):
    features = model.get_global_features(batch)
    return F.normalize(features.float(), dim=-1)


def extract_prompted_features(batch, prompt):
    prompts = [prompt] * batch.shape[0]
    features = model.get_global_features(batch, prompts)
    return F.normalize(features.float(), dim=-1)


def encode_dataset_features(prompt, progress_bar=None, status_html=None, stop_state=None):
    warning_html = ""
    if device.type != "cuda":
        warning_html = '<span style="color: orange;"><b>Warning:</b> No GPU detected. Encoding may be slow.</span>'

    if progress_bar is not None:
        progress_bar.value = 0
        progress_bar.max = len(dataloader)
        progress_bar.bar_style = "info"

    if status_html is not None:
        base_status = f'<b>Status:</b> Encoding features for prompt "{prompt}"...'
        status_html.value = f"{warning_html}<br>{base_status}" if warning_html else base_status

    base_features = []
    prompted_features = []

    with torch.inference_mode():
        for i, batch in enumerate(dataloader, start=1):
            if stop_state is not None and stop_state["stop"]:
                if progress_bar is not None:
                    progress_bar.bar_style = "warning"
                if status_html is not None:
                    base_status = "<b>Status:</b> Stopped."
                    status_html.value = f"{warning_html}<br>{base_status}" if warning_html else base_status
                return None, None, None

            batch = batch.to(device, non_blocking=True)

            base_features.append(extract_base_features(batch).cpu())
            prompted_features.append(extract_prompted_features(batch, prompt).cpu())

            if progress_bar is not None:
                progress_bar.value = i

            if status_html is not None:
                base_status = f"<b>Status:</b> Encoding... {i}/{len(dataloader)} batches"
                status_html.value = f"{warning_html}<br>{base_status}" if warning_html else base_status

    if progress_bar is not None:
        progress_bar.bar_style = "success"

    if status_html is not None:
        base_status = "<b>Status:</b> Finished."
        status_html.value = f"{warning_html}<br>{base_status}" if warning_html else base_status

    return (
        dataset,
        torch.cat(base_features, dim=0),
        torch.cat(prompted_features, dim=0),
    )


def topk_neighbors(features, reference_index, k=4):
    query = features[reference_index : reference_index + 1]
    scores = (features @ query.T).squeeze(1)
    scores[reference_index] = -1e9

    k = min(k, features.shape[0] - 1)
    top_scores, top_indices = torch.topk(scores, k=k)
    return top_indices.tolist(), top_scores.tolist()


def plot_neighbor_results(
    dataset,
    reference_index,
    steered_indices,
    steered_scores,
    base_indices,
    base_scores,
):
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))

    rows = [
        ("SteerViT\n(prompt-aware)", steered_indices, steered_scores),
        ("Base ViT\n(no prompt)", base_indices, base_scores),
    ]

    for row, (label, indices, scores) in enumerate(rows):
        ref_img = dataset.get_image(reference_index)
        axes[row, 0].imshow(ref_img)
        axes[row, 0].set_title(f"Reference")
        axes[row, 0].imshow(ref_img)
        axes[row, 0].text(
            -0.15, 0.5, label,
            transform=axes[row, 0].transAxes,
            rotation=90,
            va="center",
            ha="center",
            fontsize=20,
            fontweight="bold",
        )
        axes[row, 0].axis("off")

        for col, (idx, score) in enumerate(zip(indices, scores), start=1):
            img = dataset.get_image(idx)
            axes[row, col].imshow(img)
            axes[row, col].set_title(f"Top-{col} (sim={score:.3f})")
            axes[row, col].axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


def run_coco_retrieval_demo(reference_index, prompt, progress_bar=None, status_html=None, stop_state=None):
    assert model is not None, "Model is not loaded. Please load the model before proceeding."

    global dataset, transforms
    dataset.transform = transforms

    prompt = prompt.strip()
    if not prompt:
        raise ValueError("Prompt must not be empty.")

    dataset_result, base_features, prompted_features = encode_dataset_features(
        prompt,
        progress_bar=progress_bar,
        status_html=status_html,
        stop_state=stop_state,
    )

    if dataset_result is None:
        return

    if not (0 <= reference_index < len(dataset_result)):
        raise ValueError(f"reference_index must be in [0, {len(dataset_result) - 1}]")

    steered_indices, steered_scores = topk_neighbors(prompted_features, reference_index, k=4)
    base_indices, base_scores = topk_neighbors(base_features, reference_index, k=4)

    plot_neighbor_results(
        dataset=dataset_result,
        reference_index=reference_index,
        steered_indices=steered_indices,
        steered_scores=steered_scores,
        base_indices=base_indices,
        base_scores=base_scores,
    )


def build_coco_retrieval_demo_ui():
    if not WIDGETS_AVAILABLE:
        print("ipywidgets is not available in this environment.")
        print('Use: run_coco_retrieval_demo(reference_index=4486, prompt="the remote control")')
        return

    title = widgets.HTML(value="<h2>COCO Prompt-Conditional Nearest-Neighbor Retrieval</h2>")

    reference_index_input = widgets.BoundedIntText(
        value=DEFAULT_COCO_INDEX,
        min=0,
        max=len(dataset) - 1,
        step=1,
        description="COCO index:",
        layout=widgets.Layout(width="500px"),
        style={"description_width": "90px"},
    )

    prompt_input = widgets.Text(
        value=DEFAULT_COCO_PROMPT,
        description="Prompt:",
        layout=widgets.Layout(width="900px"),
        style={"description_width": "90px"},
        placeholder="Enter a text prompt...",
    )

    run_button = widgets.Button(
        description="Find neighbors",
        button_style="primary",
        icon="search",
    )

    stop_button = widgets.Button(
        description="Stop",
        button_style="warning",
        icon="stop",
        disabled=True,
    )

    reset_button = widgets.Button(
        description="Reset defaults",
        icon="refresh",
    )

    progress_bar = widgets.IntProgress(
        value=0,
        min=0,
        max=1,
        description="Progress:",
        layout=widgets.Layout(width="900px"),
        style={"description_width": "90px"},
    )

    status_html = widgets.HTML(value="<b>Status:</b> Idle")
    output = widgets.Output()

    def set_running_state(running):
        run_state["running"] = running
        run_button.disabled = running
        stop_button.disabled = not running
        reset_button.disabled = running

    def worker(reference_index, prompt):
        try:
            with output:
                clear_output(wait=True)

            run_coco_retrieval_demo(
                reference_index=reference_index,
                prompt=prompt,
                progress_bar=progress_bar,
                status_html=status_html,
                stop_state=stop_state,
            )
        except Exception as e:
            progress_bar.bar_style = "danger"
            status_html.value = f"<b>Status:</b> Error: {e}"
            with output:
                print(f"Error: {e}")
        finally:
            set_running_state(False)

    def on_run_clicked(_):
        if run_state["running"]:
            return

        stop_state["stop"] = False
        progress_bar.value = 0
        progress_bar.max = 1
        progress_bar.bar_style = "info"
        status_html.value = "<b>Status:</b> Preparing..."

        set_running_state(True)

        thread = threading.Thread(
            target=worker,
            args=(reference_index_input.value, prompt_input.value),
            daemon=True,
        )
        run_state["thread"] = thread
        thread.start()

    def on_stop_clicked(_):
        if not run_state["running"]:
            return

        stop_state["stop"] = True
        status_html.value = "<b>Status:</b> Stop requested..."

    def on_reset_clicked(_):
        if run_state["running"]:
            return
        reference_index_input.value = DEFAULT_COCO_INDEX
        prompt_input.value = DEFAULT_COCO_PROMPT

    run_button.on_click(on_run_clicked)
    stop_button.on_click(on_stop_clicked)
    reset_button.on_click(on_reset_clicked)

    controls = widgets.VBox([
        title,
        reference_index_input,
        prompt_input,
        widgets.HBox([run_button, stop_button, reset_button]),
        progress_bar,
        status_html,
        output,
    ])

    display(controls)


build_coco_retrieval_demo_ui()